<a href="https://colab.research.google.com/github/KawiraK/Data_wrangling_in_python/blob/main/Week2_Assignment2_Data_Wrangling_Starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 2, Assignment #2: Data Wrangling in Python

**Course:** Data & AI Specialist Program, Cyber Shujaa
**Opens:** Monday, 14 September 2026
**Due:** Sunday, 20 September 2026, 23:59
**Submit:** you do not upload this notebook or your .csv file directly. Make your finished notebook public (link only), add that link to the Assignment Submission Template, then submit the completed template as a PDF via the file submission area on the LMS

## What is data wrangling?

Data wrangling is the process of taking raw, messy data and turning it into a
clean, structured format that is actually ready to analyze. Real-world data
almost never arrives ready to use: values go missing, dates get stored as
plain text, the same category gets spelled different ways, and some records
are just wrong. Wrangling is the work of finding those problems and fixing
them, following a repeatable process: Discovery, Structuring, Cleaning,
Transformation, Validation, and Publishing, the same six steps you saw in
class this week.

## The dataset

You will wrangle the
[Netflix Shows dataset on Kaggle](https://www.kaggle.com/datasets/shivamb/netflix-shows),
a catalog of movies and TV shows on Netflix with details like director, cast,
country, release year, rating, and duration. It has plenty of real-world
messiness to work through: thousands of missing `director`, `cast`, and
`country` values, dates stored as plain text, a `duration` column that mixes
"90 min" movies with "2 Seasons" TV shows, and a handful of records where the
dates genuinely do not make sense.


## Step 1: Get the dataset onto this notebook

This assignment is done in a **Kaggle Notebook**, so the dataset comes in as
an attached input, no downloads or API tokens needed:

1. On the right-hand panel of your Kaggle Notebook, click **+ Add Input**.
2. Search for **"Netflix Shows"** and add the dataset published by
   **shivamb** (`shivamb/netflix-shows`).
3. Once it is attached, it appears under `/kaggle/input/netflix-shows/`.
   Run the cell below to confirm the file is there.


In [ ]:
import os

for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

## Step 2: Discovery - load and understand the raw data

Before changing anything, look at what you actually have. This step is pure
observation, no decisions yet: shape, column names, data types, and how much
is missing where.


In [ ]:
import pandas as pd

filepath = "/kaggle/input/netflix-shows/netflix_titles.csv"
df = pd.read_csv(filepath)

df.info()
print("Shape of the dataset (R x C):", df.shape)
print("Missing values per column:\n", df.isnull().sum())
print("Number of duplicate rows:", df.duplicated().sum())

**Answer here, in your own words, before moving on:** which three columns
have the most missing values, and roughly how many missing values does each
have? You will need this later when you decide how to handle each one.

*Your answer here.*


## Step 3: Structuring - format and standardize

Right now almost every column is stored as plain text, even the ones that
are really numbers or dates. Fix the column names first (harmless if they are
already clean), then convert `date_added` to a real datetime and split
`duration` into a numeric value plus its unit ("min" or "Season(s)").


In [ ]:
def clean_column_names(d):
    """Strips whitespace and lowercases column names, replacing spaces with underscores."""
    d.columns = d.columns.str.strip().str.lower().str.replace(" ", "_")
    return d

df = clean_column_names(df)

In [ ]:
df["date_added"] = df["date_added"].str.strip()
df["date_added"] = pd.to_datetime(df["date_added"], format="mixed")

dur = df["duration"].str.extract(r"(\d+)\s*(\w+)")
df["duration_value"] = pd.to_numeric(dur[0])
df["duration_unit"] = dur[1]

print(df[["date_added", "duration", "duration_value", "duration_unit"]].head())

## Step 4: Cleaning - duplicates and an irrelevant column

Remove exact duplicate rows, and drop the `description` column since it is
free text you will not be wrangling in this assignment.


In [ ]:
before = len(df)
df = df.drop_duplicates()
print(f"Dropped {before - len(df)} exact duplicate rows.")

df = df.drop(columns=["description"])

## Step 5: Cleaning - handling missing values

`director`, `cast`, and `country` all have real gaps. Dropping every row with
a missing value would throw away a large share of the dataset, so instead use
what else is in the row to make a reasonable guess before falling back to a
placeholder.

**Director from a repeated director/cast pair:** if the same director/cast
pairing shows up three or more times in the dataset, and one row with that
same cast is missing its director, it is a reasonable guess that the missing
director is the same one. Any director still missing after that gets labeled
`"Not Given"` rather than dropped, losing an entire row over one missing
field is not worth it here.

**Country from the director:** once `director` is filled in as much as
possible, use it the same way to fill in some of the missing `country`
values (a director's country of origin/production is a reasonable, if
imperfect, guess). Anything still missing becomes `"Not Given"`.

**Everything else:** `cast` still missing becomes `"Not Given"` too. But
`date_added`, `rating`, and `duration` are missing in only a handful of rows,
too few to be worth guessing at, so drop those rows outright instead.


In [ ]:
# Build director/cast pair lookup from pairs that repeat 3+ times with a
# real director attached.
work = df.dropna(subset=["cast"]).copy()
work["dir_cast"] = work["director"].fillna("") + "---" + work["cast"]
counts = work["dir_cast"].value_counts()
filtered = counts[counts >= 3]

dict_direcast = {}
for combo in filtered.index:
    director, cast = combo.split("---", 1)
    if director:
        dict_direcast[cast] = director

filled_directors = 0
for cast_val, director_val in dict_direcast.items():
    mask = df["director"].isna() & (df["cast"] == cast_val)
    filled_directors += int(mask.sum())
    df.loc[mask, "director"] = director_val
print(f"Filled {filled_directors} missing directors from repeated director/cast pairs.")

df.loc[df["director"].isna(), "director"] = "Not Given"

# Director -> country lookup from rows that already have a country.
dir_country = (
    df.dropna(subset=["country"])
    .drop_duplicates(subset=["director"])
    .set_index("director")["country"]
    .to_dict()
)
filled_countries = 0
for director_val, country_val in dir_country.items():
    mask = df["country"].isna() & (df["director"] == director_val)
    filled_countries += int(mask.sum())
    df.loc[mask, "country"] = country_val
print(f"Filled {filled_countries} missing countries from director lookup.")

df.loc[df["country"].isna(), "country"] = "Not Given"
df.loc[df["cast"].isna(), "cast"] = "Not Given"

before = len(df)
df = df.dropna(subset=["date_added", "rating", "duration"])
print(f"Dropped {before - len(df)} rows with missing date_added/rating/duration.")

print(df.isnull().sum())

## Step 6: Cleaning - inconsistencies and sanity checks

A row where `date_added` (when Netflix added the title) is *earlier* than
`release_year` (when the title was actually made) does not make sense.
Netflix also did not exist before 1997, so no row should be added before
that year. Find these, decide how to handle them, and sanity-check the
categorical columns while you are in there.


In [ ]:
# TODO 1: count how many rows have date_added's year earlier than
# release_year. Look at a few of them. Decide how to handle them (fix the
# year, drop the row, or something else) and explain your choice in the
# reflection section below.
10 rows.I found 10 rows where date_added predates release_year, meaning Netflix supposedly added the title before it was even made that is most were within 1 year, suggesting these may be titles with an early festival release year recorded instead of the streaming year.Since there's no reliable way to tell whether the error is in date_added or release_year without an external source to check against, I chose to drop these rows rather than guess correct one of the fields instead losing 10 rows out of several thousand is a much smaller risk than introducing an incorrect date.


# TODO 2: count how many rows have date_added before 1997 (before Netflix
# existed at all). Handle these the same way you decided above.
 0 row.After handling the date_added-vs-release_year inconsistencies in TODO 1, I found 0 rows with date_added before 1997.This makes sense for two reasons that is  any extreme date errors that would have landed before 1997 were already removed by the previous check, and this dataset's real date_added values only begin around 2008, well after Netflix's 1997 founding so this sanity check confirmed the data was clean.


# TODO 3: print df['type'].unique() and df['rating'].unique() and check the
# values look sane (no stray typos or one-off categories).



## Step 7: Transformation - split `listed_in` into separate columns

`listed_in` holds up to a few comma-separated genres per title (e.g.
"Dramas, International Movies"). Splitting it into separate columns makes
each genre analyzable on its own in a later step.


In [ ]:
# TODO: find the maximum number of comma-separated categories any single row
# has in 'listed_in', then split the column into that many new columns
# (listed_in_1, listed_in_2, ...). Rows with fewer categories should end up
# with missing values in the extra columns, that is expected.



## Step 8: Validation

Before publishing, check that the cleanup actually worked: correct data
types, no more missing values in the columns you fixed, and the data looks
right when you sample a few rows by eye.


In [ ]:
# TODO: check that date_added is a real datetime column and duration_value
# is numeric (assert statements are a good way to do this).


# TODO: confirm director and country have zero missing values left.


# TODO: reset the DataFrame's index, then print df.sample(5) to eyeball the
# result.



## Step 9: Publishing - export your cleaned dataset

Save your cleaned data as a .csv inside this notebook. You will not upload
this file separately, it stays inside your notebook (or repository) as part
of your published project.


In [ ]:
# TODO: choose a clear filename, something like "cleaned_netflix_yourname.csv".
output_filename = "Karen_Kinyua"

df.to_csv(output_filename, index=False)

# Prove the export actually worked by reading it back in.
check_df = pd.read_csv(output_filename)
print(f"Saved and reloaded {check_df.shape[0]} rows, {check_df.shape[1]} columns.")
check_df.head()

## Step 10: Reflection (required, answer in this cell)

Replace the placeholders below with your own two or three sentence answers.

1. **Which missing-value strategy did you use for `director`, and why that
   one instead of just dropping every row with a missing director?**
   
   I filled missing director values using repeated director/cast pairs  where the same director/cast pairing appeared 3 or more times with the director known, I applied that director to any row missing it, which recovered 18 directors. Anything still missing after that became "Not Given" rather than dropping the row, because those rows still have valid data in every other column (cast, country, rating, genre) dropping the whole record over one missing field would throw away information I could still use elsewhere in the analysis.

2. **What did the date-consistency checks in Step 6 turn up, and how did you
   decide to handle the rows they flagged?**
   
   The TODO 1 check found 10 rows where date_added predates release_year  meaning Netflix supposedly added the title before it was even made. Since there's no reliable way to tell whether the error is in date_added or release_year without an external source to verify against, I chose to drop these rows rather than guess correct one of the fields. The TODO 2 check then found 0 rows with date_added before 1997  this makes sense because the most extreme date errors were already removed by the first check, and this dataset's real date_added values only begin around 2008 anyway, well after Netflix's 1997 founding, so this check confirmed the data was already clean on that dimension rather than uncovering anything new.

3. **If you had another hour, what would you add or fix in this pipeline?**
   
   Split country the same way as listed_in ; it's also comma-separated for co-productions (e.g. "United States, India"), so it's currently being treated as one long string per row instead of being analyzable per country.


## Step 11: Make your notebook public, then submit your report

You do not upload this notebook or your .csv file to the LMS directly. What
you actually submit is a PDF report that links to them.

1. **Make your notebook public.** Click **Save Version** (top right), then once it has saved, open the notebook's **Settings** and set **Sharing** to **Public**. Copy the notebook's URL, that is your public link.
2. **Open the Assignment Submission Template** (in the General section of
   this course) and fill it in following its own sections.
3. **Paste your public notebook link into the template.**
4. **Export the completed template as a PDF**, and submit that PDF using the
   file submission area on the LMS. That PDF, with your link inside it, is
   your actual submission.


## Submission checklist

- [ ] Step 3's `date_added` conversion and `duration` split both actually ran without errors
- [ ] Step 4 removed duplicate rows and the `description` column
- [ ] Step 5 shows evidence of the director/cast/country fill logic actually running, not just "Not Given" everywhere
- [ ] Step 6 reports how many inconsistent-date rows you found and states how you handled them
- [ ] Step 7's `listed_in` split produced the right number of category columns
- [ ] Step 8's validation checks pass (dtypes correct, no missing values left in director/country)
- [ ] The exported .csv from Step 9 opens correctly when reloaded
- [ ] All three reflection questions in Step 10 are answered in your own words
- [ ] Your notebook link is genuinely public (test it in a private/incognito browser window)
- [ ] Your public link is pasted into the Assignment Submission Template
- [ ] You have submitted the completed template as a PDF on the LMS

**Grading scale:** 0 = No submission, 1 = Below Expectation, 2 = Meets Expectation,
3 = Exceeds Expectation.